## PMN Distance to Immune cells

#### Load Protein Data

In [1]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np

# Load files on Evan's Laptop
# expr_orig = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\expression.csv", index_col=0)
# expr=expr_orig.transpose()
# metadata = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\metadata.csv", index_col=0)
# umap = pd.read_csv(r"C:\Users\evanj\OneDrive\Documents\umap.csv", index_col=0)

#Load files on Lab computer
expr_orig = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\expression.csv", index_col=0)
expr=expr_orig.transpose()
metadata = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\metadata.csv", index_col=0)
umap = pd.read_csv(r"C:\Users\ejohns\Documents\Shapiro Data Files\umap.csv", index_col=0)

# Create AnnData object
adata = sc.AnnData(X=expr.values)

# Assign metadata
adata.obs = metadata
adata.var_names = expr.columns
adata.obs_names = expr.index

# Add spatial coordinates and UMAP to .obsm
# adata.obsm["spatial"] = metadata[['x_FOV_px', 'y_FOV_px']].values  # adjust if needed
adata.obsm["spatial"] = metadata[['x_FOV_px']].assign(y_FOV_px = -metadata['y_FOV_px']).values
adata.obsm["X_umap"] = umap.values

C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
C:\Users\ejohns\AppData\Local\anaconda3\Lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import annd

### Exact Distances of PMN cells to other cell types 

This is probably not very helpful and not particullary useful in and of itself. However, this was requested by Huy so I will go along with it. What this does is measure the distance between PMN and the targeted cell types (more can be added easily) and this will return information like the average distance to the closest cell and the average distance to all cells. 

In [13]:
# Output. Each sample in order with the following columns Sample ID, PMN Count, Resonder Status, Average Min Distance CD4, Average Min Distance  CD8,  Average Min Distance  tumor cells,  Average Min Distance Treg (to start
import pandas as pd
import importlib
import my_functions
importlib.reload(my_functions)

# Define column names
columns = ['Sample ID', 'PMN Count', 'Responder Status', 'Avg Min CD4 Distance', 'Avg Min CD8 Distance', 'Avg Min Treg Distance', 'Average Min Tumor Distance', 'Avg CD4 Distance', 'Avg CD8 Distance', 'Avg Treg Distance', 'Average Tumor Distance']

# Create empty DataFrame with those columns
summary_matrix = pd.DataFrame(columns=columns)

for i in range(1,4):
    for j in range(1,26):
        
        # Data to add to matrix
        sample_id=f"c_{i}_{j}_"
        pmn_count=my_functions.pmn_counter(adata,sample_id)["PMN Count"]

        # Must have PMN cells - Exclude cases without pmn cells
        if pmn_count<1:
            continue
        responder_status=my_functions.get_sample_info(adata,sample_id)["Response"]

        # CD4 Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"CD4+T_cells")["Cell Count"] > 0:
            results_CD4=my_functions.nearest_cells_of_particular_type(adata,sample_id,"CD4+T_cells")
            avg_min_CD4=results_CD4["Average Mininum Distance"]
            avg_CD4=results_CD4["Average Distance"]
        else:
            avg_min_CD4=0
            avg_CD4=0

        # CD8 Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"CD8+T_cells")["Cell Count"] > 0:
            results_CD8=my_functions.nearest_cells_of_particular_type(adata,sample_id,"CD8+T_cells")
            avg_min_CD8=results_CD8["Average Mininum Distance"]
            avg_CD8=results_CD8["Average Distance"]
        else:
            avg_min_CD8=0
            avg_CD8=0

        # Treg Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"Treg")["Cell Count"] > 0:
            results_Treg=my_functions.nearest_cells_of_particular_type(adata,sample_id,"Treg")
            avg_min_Treg=results_Treg["Average Mininum Distance"]
            avg_Treg=results_Treg["Average Distance"]
        else:
            avg_min_Treg=0
            avg_Treg=0

        # Tumor Cell Counts
        if my_functions.matching_cell_list(adata,sample_id,"Tumor_cells")["Cell Count"] > 0:
            results_Tumor=my_functions.nearest_cells_of_particular_type(adata,sample_id,"Tumor_cells")
            avg_min_tumor=results_Tumor["Average Mininum Distance"]
            avg_tumor=results_Tumor["Average Distance"]
        else:
            avg_min_tumor=0
            avg_tumor=0
            
        # Must have PMN cells - Exclude cases without pmn cells
        if pmn_count<1:
            continue

        # Adding data to matrix
        summary_matrix.loc[len(summary_matrix)] = [sample_id, pmn_count, responder_status, avg_min_CD4, avg_min_CD8, avg_min_Treg, avg_min_tumor, avg_CD4, avg_CD8, avg_Treg, avg_tumor]
        
display(summary_matrix)
summary_matrix.to_excel("summary_matrix_2.xlsx", index=False)

[NbConvertApp] Converting notebook my_functions.ipynb to python
[NbConvertApp] Writing 24140 bytes to my_functions.py


,Sample ID,PMN Count,Responder Status,Avg Min CD4 Distance,Avg Min CD8 Distance,Avg Min Treg Distance,Average Min Tumor Distance,Avg CD4 Distance,Avg CD8 Distance,Avg Treg Distance,Average Tumor Distance
0,c_1_1_,55,R,273.972614,592.759764,631.634319,11.090332,525.352298,735.917035,651.116774,446.532664
1,c_1_3_,6,R,28.753049,170.031600,106.209140,11.824233,246.332199,351.629350,224.877578,330.041189
2,c_1_4_,12,R,52.528240,99.802765,73.496580,23.712366,234.948735,287.666555,248.699919,369.395953
3,c_1_5_,15,R,86.062535,150.281274,64.924672,15.375530,347.959190,392.597713,366.510778,400.521354
4,c_1_6_,23,R,19.640478,375.802924,74.626134,33.386715,99.011878,573.904537,74.626134,496.276498
...,...,...,...,...,...,...,...,...,...,...,...
61,c_3_16_,18,NR,75.778988,183.079156,64.614820,29.833396,302.747329,264.613239,294.534193,463.074163
62,c_3_17_,22,NR,63.788160,151.555074,100.891480,30.778431,336.877326,260.473369,317.286085,496.839089
63,c_3_18_,36,NR,55.022038,144.141265,64.205251,28.126310,419.253659,459.742630,409.799740,474.083958
64,c_3_19_,36,NR,48.176947,348.449298,62.653424,22.721432,370.779279,513.769397,328.502648,404.308327


In [31]:
summary_matrix.to_excel("summary_matrix_2.xlsx", index=False)

In [7]:
import my_functions
import importlib
importlib.reload(my_functions)


[NbConvertApp] Converting notebook my_functions.ipynb to python
[NbConvertApp] Writing 24203 bytes to my_functions.py


<module 'my_functions' from 'C:\\Users\\ejohns\\Documents\\GitHub\\2025-Dinh-Lab-Research-Project\\Shapiro 2025 Project\\Python SquidPy\\my_functions.py'>